# Scotch Whisky: Consumer Progression & Premiumization Intelligence

A product intelligence case study looking at how shoppers move from accessible Scotch into more distinctive premium expressions, and where shelf context, gifting, and price steps help or interrupt that journey.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('dark_background')
base = Path.cwd()
df = pd.read_csv(base / 'data' / 'scotch_progression_observations.csv') if (base / 'data').exists() else pd.read_csv(base / 'scotch-whisky-consumer-progression-intelligence' / 'data' / 'scotch_progression_observations.csv')
df.head()


## 1. Dataset Overview & Imperfection Notes

**Question:** Does this still feel like shelf reality, or did we accidentally clean the life out of it?

The case keeps missing occasion and notes fields, rough shelf text, and imperfect repeat signals on purpose. It should feel like something assembled from retail observation, not a polished CRM export.


In [ ]:
sample = df[['brand','expression','tier','bottle_price','price_band','purchase_type','repeat_purchase_flag']].sample(10, random_state=42)
sample

imperfections = pd.DataFrame({
    'metric': ['rows', 'missing occasion %', 'missing notes %', 'gift boxed share %', 'repeat share %'],
    'value': [len(df), round(df['occasion'].isna().mean()*100,1), round(df['notes_text'].isna().mean()*100,1), round(df['is_gift_boxed'].mean()*100,1), round(df['repeat_purchase_flag'].mean()*100,1)]
})
imperfections


## 2. Product Tier Distribution

**Question:** Where does the category still live day to day, and where does value start to thin out?

Entry and core should still hold most of the weekly traffic. Premium and distinctive matter because they change value, preference, and confidence, not because they overpower the category by count.


In [ ]:
tier_summary = df.groupby('tier').agg(observations=('brand','size'), avg_price=('bottle_price','mean')).reindex(['entry','core','premium','distinctive'])
tier_summary
fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(tier_summary.index, tier_summary['observations'], color=['#3d9970','#74b67a','#d8b15c','#b47a44'])
ax.set_title('Tier distribution')
ax.set_ylabel('Observations')
plt.show()


## 3. Price Step & Progression Friction

**Question:** At what point do shoppers look interested in trading up, but stop looking comfortable doing it repeatedly? The sharpest friction shows up around R320-R420.

The useful read is where curiosity keeps climbing without repeat climbing with it. That is usually where premiumization is visible, but not yet comfortable.


In [ ]:
price_bins = pd.cut(df['bottle_price'], bins=[0,320,420,520,700,950,1400,1800], include_lowest=True)
price_friction = df.assign(price_step=price_bins.astype(str)).groupby('price_step').agg(repeat_rate=('repeat_purchase_flag','mean'), exploratory_share=('estimated_buyer_stage', lambda s: s.isin(['exploratory','preference_led']).mean()), observations=('brand','size')).reset_index()
price_friction
fig, ax = plt.subplots(figsize=(9,4.6))
ax.plot(price_friction['price_step'], price_friction['repeat_rate']*100, marker='o', label='Repeat rate')
ax.plot(price_friction['price_step'], price_friction['exploratory_share']*100, marker='o', label='Exploratory or preference-led share')
ax.legend()
plt.xticks(rotation=20)
plt.show()


## 4. Packaging & Gifting Behaviour

**Question:** Does gift packaging recruit people into premium Scotch, or does it just create a good one-off occasion?

Gift packaging can widen the top of the funnel without earning a lasting place in the shopper's own routine. That difference matters.


In [ ]:
packaging = df.groupby('is_gift_boxed').agg(gift_share=('purchase_type', lambda s: (s == 'gift').mean()), repeat_rate=('repeat_purchase_flag','mean'), avg_price=('bottle_price','mean')).reset_index()
packaging['segment'] = packaging['is_gift_boxed'].map({0:'not gift boxed',1:'gift boxed'})
packaging


## 5. Retail Display Influence

**Question:** Which retail environments help discovery, and which ones actually hold repeat?

Feature displays should help bottles get noticed. The standard shelf is where preference often shows itself with less theatre.


In [ ]:
display = df.groupby('display_type').agg(discovery_share=('estimated_buyer_stage', lambda s: s.isin(['entry','exploratory']).mean()), repeat_rate=('repeat_purchase_flag','mean'), promo_share=('promo_flag','mean')).reset_index()
display


## 6. Repeat Purchase vs Trial Behaviour

**Question:** Which products are opening the door, and which ones look like places people eventually settle? Gateway: Johnnie Walker Black Label. Destination: Lagavulin 16.

Gateway products still need enough self-purchase comfort to recruit buyers. Destination products matter because a smaller set of shoppers keep coming back to them.


In [ ]:
product_view = df.assign(product_key=df['brand'] + ' ' + df['expression'].astype(str)).groupby('product_key').agg(observations=('brand','size'), repeat_rate=('repeat_purchase_flag','mean'), avg_price=('bottle_price','mean'), self_share=('purchase_type', lambda s: (s == 'self').mean())).sort_values('observations', ascending=False)
product_view


## 7. Progression Signals & Stalling Points

**Question:** Where do we see genuine movement upward, and where do shoppers appear to hesitate, bounce, or need support? Key read: Gateway bottles are not just cheaper; they are the ones that still look safe enough for self-purchase while nudging shoppers into a clearer style cue or price step.

Not all movement upward is healthy. Some bottles create curiosity without conviction, while others quietly build a smaller but stronger preference base.


In [ ]:
stall = df.copy()
stall['progression_state'] = np.select([(stall['tier'].isin(['entry','core'])) & (stall['estimated_buyer_stage'] == 'entry'), (stall['tier'].isin(['premium','distinctive'])) & (stall['repeat_purchase_flag'] == 0), (stall['tier'].isin(['premium','distinctive'])) & (stall['repeat_purchase_flag'] == 1), (stall['promo_flag'] == 1) & (stall['tier'].isin(['core','premium']))], ['Stayed accessible','Tried up, did not return','Traded up and returned','Needed promo support'], default='Other')
stall['progression_state'].value_counts()


## 8. Key Commercial Insights

**Question:** What should a commercial or category team actually do with this?

The category still recruits through familiar entry points, but loyalty sits closer to confidence than promotion. Gift packaging helps reach, standard shelf helps repeat, and the price ladder needs clearer hand-holds once the stretch becomes visible.
